# FLAIR Group 4: Multimodal Mood and Theme Tagging of Music
**Late fusion of pre-trained audio features and track text metadata on MTG-Jamendo (mood/theme, split-0)**

This notebook reproduces every number reported in the paper. Its sections follow the paper:

| Notebook section | Paper section |
|---|---|
| 1. Setup and requirements | 4.3 Protocol (reproducibility) |
| 2. Dataset and official split | 4.1 Dataset and Preprocessing |
| 3. Track text metadata | 3.2 Text branch, 4.1 |
| 4. Tag-frequency groups and evaluation functions | 4.2 Baselines and Evaluation |
| 5. Text-only baseline and tag-frequency analysis | 3.3 Text branch, 5 |
| 6. Audio branch (MusiCNN embeddings) | 3.2 Audio branch |
| 7. Audio-only baseline and late fusion | 3.3 Late fusion, 4.2 |
| 8. Success criteria and final test evaluation | 4.2, 4.3, 5 |

**Run switches** (Section 1): `RUN_AUDIO` downloads the audio and extracts MusiCNN embeddings (long-running; see Section 6).
`RUN_FINAL_TEST` evaluates on the held-out test split and must only be switched on once, after all model choices are frozen.

## 1. Setup and requirements

In [1]:
# Requirements. Packages that affect the results are pinned; numpy, pandas and joblib use the versions
# preinstalled in Colab (pinning them would force a runtime restart). The exact versions of everything
# used in a run are printed in the next cells and saved to run_info.json.
#   scikit-learn==1.6.1          TF-IDF, logistic regression, metrics
#   essentia-tensorflow==2.1b6.dev1389   MusiCNN audio embeddings (audio branch only)
%pip install -q scikit-learn==1.6.1 essentia-tensorflow==2.1b6.dev1389

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports, run switches, paths, and the fixed random seed used everywhere.
import os, sys, csv, json, hashlib, platform, shutil, subprocess, tarfile, urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.stats import spearmanr

RUN_AUDIO = True        # True = download audio + extract MusiCNN embeddings (hours; needs ~5 GB free disk at a time)
RUN_FINAL_TEST = False   # True = one-time evaluation on the held-out test split, after development is finished

SEED = 42
LR_C = 1.0   # inverse L2-regularisation strength, identical for the audio and text classifiers
np.random.seed(SEED)

# Works in Colab (/content) and locally (current folder)
BASE_DIR = '/content' if os.path.isdir('/content') else os.getcwd()
REPO_DIR = os.path.join(BASE_DIR, 'mtg-jamendo-dataset')
DATA_DIR = os.path.join(REPO_DIR, 'data')
SPLIT_DIR = os.path.join(DATA_DIR, 'splits', 'split-0')
OUTPUT_DIR = os.path.join(BASE_DIR, 'FLAIR_Group_4_results')
EMB_DIR = os.path.join(OUTPUT_DIR, 'musicnn_embeddings')
os.makedirs(EMB_DIR, exist_ok=True)

# Exact dataset version used in the paper (MTG-Jamendo GitHub commit)
DATASET_COMMIT = 'cafd8e20c265ed84f1e61f1c875327971f43a62f'

print('Run switches -> RUN_AUDIO:', RUN_AUDIO, '| RUN_FINAL_TEST:', RUN_FINAL_TEST)
print('Base directory:', BASE_DIR)

Run switches -> RUN_AUDIO: True | RUN_FINAL_TEST: False
Base directory: /mnt/d/FLAIR-Group-4/flair-mood-tagging


In [3]:
# Record the software versions actually used for this run (copied into the paper, Section 4.3).
versions = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'joblib': joblib.__version__,
}
try:
    import essentia
    versions['essentia'] = essentia.__version__
except ImportError:
    versions['essentia'] = 'not installed (only needed when RUN_AUDIO = True)'
for name, version in versions.items():
    print(f'{name:>13}: {version}')

       python: 3.12.3
        numpy: 2.5.4
       pandas: 3.1.1
 scikit-learn: 1.6.1
       joblib: 1.6.0
     essentia: 2.1-beta6-dev


## 2. Dataset and official split
MTG-Jamendo mood/theme subset (Bogdanov et al. 2019). We clone the dataset repository at a fixed commit,
load the mood/theme annotations, and use the official **split-0** train/validation/test partitions.

In [4]:
# Clone the MTG-Jamendo repository once and check out the exact commit used in the paper.
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/MTG/mtg-jamendo-dataset.git', REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', DATASET_COMMIT], check=True)
head = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('MTG-Jamendo commit:', head)

MTG-Jamendo commit: cafd8e20c265ed84f1e61f1c875327971f43a62f


In [5]:
def read_tag_file(path):
    # Annotation files are tab-separated; the TAGS column holds a variable number of tags,
    # so everything after the fifth column is collected into one list.
    rows = []
    with open(path, encoding='utf-8') as f:
        next(f)                                    # skip header
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip('\n').split('\t')
            rows.append(parts[:5] + [parts[5:]])
    return pd.DataFrame(rows, columns=['TRACK_ID', 'ARTIST_ID', 'ALBUM_ID', 'PATH', 'DURATION', 'TAGS'])

# Full mood/theme annotation file
df = read_tag_file(os.path.join(DATA_DIR, 'autotagging_moodtheme.tsv'))
tag_counts = Counter(tag for tags in df['TAGS'] for tag in tags)
print('Full annotation file:', df.shape[0], 'tracks,', len(tag_counts), 'unique tags')
display(df.head())

Full annotation file: 18486 tracks, 59 unique tags


,TRACK_ID,ARTIST_ID,ALBUM_ID,PATH,DURATION,TAGS
0,track_0000948,artist_000087,album_000149,48/948.mp3,212.7,[mood/theme---background]
1,track_0000950,artist_000087,album_000149,50/950.mp3,248.0,[mood/theme---background]
2,track_0000951,artist_000087,album_000149,51/951.mp3,199.7,[mood/theme---background]
3,track_0002165,artist_000326,album_000347,65/2165.mp3,229.0,[mood/theme---film]
4,track_0002263,artist_000320,album_000366,63/2263.mp3,494.7,[mood/theme---melancholic]


In [6]:
# Official split-0 partitions
splits = {s: read_tag_file(os.path.join(SPLIT_DIR, f'autotagging_moodtheme-{s}.tsv'))
          for s in ['train', 'validation', 'test']}
for s, d in splits.items():
    print(f'{s:>10}: {len(d):>5} tracks')
print(f'{"total":>10}: {sum(len(d) for d in splits.values()):>5} tracks')

     train:  9949 tracks
validation:  3802 tracks
      test:  4231 tracks
     total: 17982 tracks


In [7]:
# Leakage checks: tracks, artists and albums must not appear in more than one partition.
# This matters because the text input contains the artist and album names.
pairs = [('train', 'validation'), ('train', 'test'), ('validation', 'test')]
for column in ['TRACK_ID', 'ARTIST_ID', 'ALBUM_ID']:
    overlaps = {f'{a}/{b}': len(set(splits[a][column]) & set(splits[b][column])) for a, b in pairs}
    print(f'{column:>9} overlap:', overlaps)
    assert all(v == 0 for v in overlaps.values()), f'{column} overlap between partitions!'
print('No track, artist, or album appears in more than one partition.')

 TRACK_ID overlap: {'train/validation': 0, 'train/test': 0, 'validation/test': 0}
ARTIST_ID overlap: {'train/validation': 0, 'train/test': 0, 'validation/test': 0}
 ALBUM_ID overlap: {'train/validation': 0, 'train/test': 0, 'validation/test': 0}
No track, artist, or album appears in more than one partition.


In [8]:
# The official splits keep 56 of the 59 tags in the full file.
official_tags = sorted({t for d in splits.values() for tags in d['TAGS'] for t in tags})
print('Official tags in split-0:', len(official_tags))
print('Tags only in the full file:', sorted(set(tag_counts) - set(official_tags)))

# Binary label matrices (tracks x 56 tags), always in the same alphabetical tag order
mlb = MultiLabelBinarizer(classes=official_tags)
mlb.fit([official_tags])
Y = {s: mlb.transform(d['TAGS']) for s, d in splits.items()}
for s in Y:
    print(f'{s:>10}: label matrix {Y[s].shape}, tracks without a tag: {(Y[s].sum(axis=1) == 0).sum()}')

Official tags in split-0: 56
Tags only in the full file: ['mood/theme---ambiental', 'mood/theme---horror', 'mood/theme---mellow']
     train: label matrix (9949, 56), tracks without a tag: 0
validation: label matrix (3802, 56), tracks without a tag: 0
      test: label matrix (4231, 56), tracks without a tag: 0


## 3. Track text metadata
`raw.meta.tsv` contains the track title, artist name and album name. Some album names begin with a double quote,
so the file is parsed **line by line** (no CSV quoting rules). This loads every row directly, with no recovery step.

In [9]:
def load_metadata(path):
    # Columns: TRACK_ID, ARTIST_ID, ALBUM_ID, TRACK_NAME, ARTIST_NAME, ALBUM_NAME, RELEASEDATE, URL.
    # A few rows contain an extra (empty) tab inside the text fields; those empty pieces are dropped.
    records, irregular = [], 0
    with open(path, encoding='utf-8') as f:
        next(f)
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip('\n').split('\t')
            text_fields = parts[3:-2]                     # title, artist, album
            if len(text_fields) != 3:
                irregular += 1
                text_fields = [t for t in text_fields if t != ''] if len(text_fields) > 3 else text_fields
                if len(text_fields) > 3:                  # any remaining extra tab belongs to the album name
                    text_fields = text_fields[:2] + ['\t'.join(text_fields[2:])]
            records.append({'TRACK_ID': parts[0], 'TRACK_NAME': text_fields[0],
                            'ARTIST_NAME': text_fields[1], 'ALBUM_NAME': text_fields[2]})
    return pd.DataFrame(records), irregular

raw_lines = sum(1 for _ in open(os.path.join(DATA_DIR, 'raw.meta.tsv'), encoding='utf-8')) - 1
metadata, irregular = load_metadata(os.path.join(DATA_DIR, 'raw.meta.tsv'))
print('Lines in raw.meta.tsv:', raw_lines, '| rows loaded:', len(metadata), '| rows with extra tabs:', irregular)
assert len(metadata) == raw_lines and metadata['TRACK_ID'].is_unique

Lines in raw.meta.tsv: 56639 | rows loaded: 56639 | rows with extra tabs: 2


In [10]:
# Join text to every track by TRACK_ID and build one text string per track: title + artist + album.
text_lookup = metadata.set_index('TRACK_ID')
for s, d in splits.items():
    t = text_lookup.loc[d['TRACK_ID']]
    d['TEXT_INPUT'] = (t['TRACK_NAME'] + ' ' + t['ARTIST_NAME'] + ' ' + t['ALBUM_NAME']).str.strip().values
    print(f'{s:>10}: missing metadata = {t.isna().any(axis=1).sum()}, empty text = {(d["TEXT_INPUT"] == "").sum()}')
display(splits['train'][['TRACK_ID', 'TEXT_INPUT', 'TAGS']].head())

     train: missing metadata = 0, empty text = 0
validation: missing metadata = 0, empty text = 0
      test: missing metadata = 0, empty text = 0


,TRACK_ID,TEXT_INPUT,TAGS
0,track_0000948,Victory Alexander Blu May,[mood/theme---background]
1,track_0000950,May Alexander Blu May,[mood/theme---background]
2,track_0000951,Nostalgia Alexander Blu May,[mood/theme---background]
3,track_0002165,Le nettoyeur Greg Baumont Wood,[mood/theme---film]
4,track_0002263,On the other side Sköhl Offensrüh First Act.,[mood/theme---melancholic]


## 4. Tag-frequency groups and evaluation functions
**Uncommon tags** = the bottom 25% of the 56 tags by number of positive training examples (14 tags).
Ties are broken alphabetically (stable sort over the alphabetical tag list).

Metrics (paper Section 4.2):
* **PR-AUC-macro**: average precision (area under the precision–recall curve) per tag, averaged with equal weight over tags. Primary metric, as in MediaEval.
* **PR-AUC-micro**: average precision over all track–tag decisions pooled together.
* **ROC-AUC-macro**: per-tag ROC-AUC averaged over tags (MediaEval's second official metric).
* **Uncommon-tag PR-AUC**: PR-AUC-macro over the 14 uncommon tags only.

In [11]:
train_counts = Y['train'].sum(axis=0)
order = np.argsort(train_counts, kind='stable')          # rarest first; ties stay alphabetical
N_UNCOMMON = len(official_tags) // 4                       # bottom 25% of 56 tags = 14
uncommon_idx = np.sort(order[:N_UNCOMMON])
common_idx = np.sort(order[N_UNCOMMON:])

short = lambda t: t.replace('mood/theme---', '')
print('Uncommon tags (training count):')
print(', '.join(f'{short(official_tags[i])} ({train_counts[i]})' for i in order[:N_UNCOMMON]))
print('Training-count cut-off: uncommon <=', train_counts[order[N_UNCOMMON - 1]],
      '| next tag has', train_counts[order[N_UNCOMMON]])

Uncommon tags (training count):
sexy (59), fast (62), groovy (78), travel (89), heavy (98), holiday (98), powerful (106), funny (109), party (127), hopeful (137), nature (137), retro (139), children (152), trailer (159)
Training-count cut-off: uncommon <= 159 | next tag has 169


In [12]:
def evaluate(y_true, probs):
    # All four metrics reported in the paper for one model on one partition.
    return {
        'PR-AUC-macro': average_precision_score(y_true, probs, average='macro'),
        'PR-AUC-micro': average_precision_score(y_true, probs, average='micro'),
        'ROC-AUC-macro': roc_auc_score(y_true, probs, average='macro'),
        'PR-AUC-macro (uncommon 14)': average_precision_score(y_true[:, uncommon_idx], probs[:, uncommon_idx], average='macro'),
        'PR-AUC-macro (other 42)': average_precision_score(y_true[:, common_idx], probs[:, common_idx], average='macro'),
    }

def group_pr_auc(y_true, probs, idx):
    # Macro PR-AUC over a group of tags, skipping tags with no positives in this (re)sample.
    scores = [average_precision_score(y_true[:, j], probs[:, j]) for j in idx if y_true[:, j].sum() > 0]
    return float(np.mean(scores))

def paired_bootstrap(y_true, probs_a, probs_b, idx, n_boot=1000, seed=SEED):
    # Paired bootstrap over tracks: resample the same tracks for both models and record the
    # difference in uncommon-tag PR-AUC (model A minus model B). Returns observed diff and 95% CI.
    rng = np.random.default_rng(seed)
    n = y_true.shape[0]
    diffs = []
    for _ in range(n_boot):
        s = rng.integers(0, n, n)
        diffs.append(group_pr_auc(y_true[s], probs_a[s], idx) - group_pr_auc(y_true[s], probs_b[s], idx))
    observed = group_pr_auc(y_true, probs_a, idx) - group_pr_auc(y_true, probs_b, idx)
    low, high = np.percentile(diffs, [2.5, 97.5])
    return observed, low, high

## 5. Text-only baseline
TF-IDF (unigrams + bigrams, max 10,000 features, min document frequency 2) fitted on **training text only**,
then one-vs-rest logistic regression (L2 regularisation, C = 1.0, liblinear, max 1,000 iterations, random state 42).

In [13]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_text = {'train': tfidf.fit_transform(splits['train']['TEXT_INPUT'])}      # vocabulary learned on train only
for s in ['validation', 'test']:
    X_text[s] = tfidf.transform(splits[s]['TEXT_INPUT'])
print({s: X.shape for s, X in X_text.items()}, '| vocabulary size:', len(tfidf.vocabulary_))

text_model = OneVsRestClassifier(
    LogisticRegression(C=LR_C, penalty='l2', solver='liblinear', max_iter=1000, random_state=SEED), n_jobs=-1)
text_model.fit(X_text['train'], Y['train'])

P_text_val = text_model.predict_proba(X_text['validation'])
text_val_results = evaluate(Y['validation'], P_text_val)
print('\nText-only baseline, VALIDATION:')
for k, v in text_val_results.items():
    print(f'  {k:<28} {v:.4f}')

{'train': (9949, 10000), 'validation': (3802, 10000), 'test': (4231, 10000)} | vocabulary size: 10000



Text-only baseline, VALIDATION:
  PR-AUC-macro                 0.1163
  PR-AUC-micro                 0.1210
  ROC-AUC-macro                0.6281
  PR-AUC-macro (uncommon 14)   0.0991
  PR-AUC-macro (other 42)      0.1220


In [14]:
# Per-tag validation results, sorted from rarest to most common training tag
per_tag = pd.DataFrame({
    'Tag': [short(t) for t in official_tags],
    'Training count': train_counts,
    'Validation count': Y['validation'].sum(axis=0),
    'Uncommon': [i in set(uncommon_idx) for i in range(len(official_tags))],
    'Text AP': [average_precision_score(Y['validation'][:, j], P_text_val[:, j]) for j in range(len(official_tags))],
}).sort_values('Training count').reset_index(drop=True)
display(per_tag.round(4))

,Tag,Training count,Validation count,Uncommon,Text AP
0,sexy,59,37,True,0.0137
1,fast,62,26,True,0.0114
2,groovy,78,74,True,0.0343
3,travel,89,49,True,0.0195
4,holiday,98,35,True,0.3650
5,heavy,98,33,True,0.1254
6,powerful,106,47,True,0.0170
7,funny,109,23,True,0.0163
8,party,127,84,True,0.0645
9,hopeful,137,36,True,0.0111


### 5.1 Text-only results by tag frequency (paper Section 5)
Average precision depends on how common a tag is: a random ranking scores about the tag's **prevalence**
(share of validation tracks with the tag). We therefore also report **lift** = AP / prevalence, and test whether
the text model's AP and lift are related to the number of training examples (Spearman rank correlation).

In [15]:
prevalence = Y['validation'].mean(axis=0)
text_ap = np.array([average_precision_score(Y['validation'][:, j], P_text_val[:, j]) for j in range(len(official_tags))])
lift = text_ap / prevalence

# Frequency quartiles of the 56 tags (quartile 1 = the 14 uncommon tags)
quartiles = np.array_split(order, 4)
quartile_table = pd.DataFrame([{
    'Quartile': q + 1,
    'Training examples': f'{train_counts[g].min()}-{train_counts[g].max()}',
    'Random AP (prevalence)': prevalence[g].mean(),
    'Text AP': text_ap[g].mean(),
    'Median lift': np.median(lift[g]),
} for q, g in enumerate(quartiles)])
display(quartile_table.round(4))

rho_ap = spearmanr(train_counts, text_ap)
rho_lift = spearmanr(train_counts, lift)
print(f'Spearman(training count, text AP)  rho = {rho_ap.correlation:.3f}, p = {rho_ap.pvalue:.3f}')
print(f'Spearman(training count, lift)     rho = {rho_lift.correlation:.3f}, p = {rho_lift.pvalue:.3f}')
print('Tags with lift >= 3:', int((lift >= 3).sum()), '| of which uncommon:', int((lift[uncommon_idx] >= 3).sum()))

,Quartile,Training examples,Random AP (prevalence),Text AP,Median lift
0,1,59-159,0.0139,0.0991,1.8332
1,2,169-264,0.0224,0.0732,2.3466
2,3,267-404,0.0256,0.1531,2.5191
3,4,415-927,0.0566,0.1398,1.7330


Spearman(training count, text AP)  rho = 0.425, p = 0.001
Spearman(training count, lift)     rho = -0.030, p = 0.824
Tags with lift >= 3: 18 | of which uncommon: 3


In [16]:
# Which words drive the strongest text predictions? Top TF-IDF terms by logistic-regression weight.
vocab = np.array(tfidf.get_feature_names_out())
top_tags = [official_tags[j] for j in np.argsort(-text_ap)[:6]]
for tag in top_tags + ['mood/theme---happy', 'mood/theme---sexy']:
    j = official_tags.index(tag)
    weights = text_model.estimators_[j].coef_.ravel()
    print(f'{short(tag):>10} (AP {text_ap[j]:.3f}): {", ".join(vocab[np.argsort(-weights)[:6]])}')

 christmas (AP 0.744): christmas, weihnachtsmusik, gemafreie, advent, night, bells
   trailer (AP 0.589): epic, trailer, paio, muswaystudio, aufklarung orchestral, orchestral loops
   holiday (AP 0.365): christmas, advent, studios, studios advent, degumoth, degumoth studios
      epic (AP 0.341): epic, trailer, anima, teichert, marc teichert, lourme
 corporate (AP 0.338): corporate, nicoco, inspiration, stereoresonance, news, inspiring
     space (AP 0.290): space, eterosmos, xcyril, fortadelis, part, xcyril organique
     happy (AP 0.243): happy, mfym, muswaystudio, ukulele, positive, opus
      sexy (AP 0.014): fashion, fashion theme, lisenkov fashion, fashion show, threadbare, show


## 6. Audio branch: MusiCNN embeddings
* **Audio:** `autotagging_moodtheme/audio-low` from the official MTG-Jamendo mirror (mono MP3, LAME VBR 2; 46 GB in 100 TAR archives, vs. 152 GB for full quality).
* **Model:** Essentia `msd-musicnn-1` (MusiCNN trained on the Million Song Dataset). Input: mono audio at 16 kHz, analysed in 3-second patches.
* **Features:** the 200-dimensional output of the penultimate layer `model/dense/BiasAdd` for each patch, **averaged over all patches of the full track** → one 200-d vector per track.

The audio is processed one TAR archive at a time (download → verify checksum → extract → embed → delete), so only ~0.5 GB of audio is on disk at once.
Embeddings are saved per archive, so the step can be resumed if Colab disconnects. Only the 17,982 split-0 tracks are embedded.

In [17]:
MODEL_PB = os.path.join(BASE_DIR, 'msd-musicnn-1.pb')
AUDIO_TYPE = 'audio-low'
MIRROR = 'https://cdn.freesound.org/mtg-jamendo/autotagging_moodtheme/' + AUDIO_TYPE + '/'
TAR_LIST = os.path.join(DATA_DIR, 'download', f'autotagging_moodtheme_{AUDIO_TYPE}_sha256_tars.txt')
AUDIO_TMP = os.path.join(BASE_DIR, 'audio_tmp')
MAX_ARCHIVES = None          # 1 = smoke test on the first of 100 archives; set to None to process all of them
EXCERPT_SECONDS = None    # None = embed the full track; e.g. 30 = embed only the middle 30 s (much faster)

split_ids = set().union(*[set(d['TRACK_ID']) for d in splits.values()])

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def embed_archive(tar_name, expected_sha, extractor, loader_cls):
    # Download one TAR, check it, embed every split-0 track inside it, then delete the audio.
    out_file = os.path.join(EMB_DIR, tar_name.replace('.tar', '.npz'))
    if os.path.exists(out_file):
        return 'already done'
    os.makedirs(AUDIO_TMP, exist_ok=True)
    tar_path = os.path.join(AUDIO_TMP, tar_name)
    urllib.request.urlretrieve(MIRROR + tar_name, tar_path)
    assert sha256(tar_path) == expected_sha, f'Checksum mismatch for {tar_name}'
    ids, embs = [], []
    with tarfile.open(tar_path) as tar:
        for member in tar.getmembers():
            if not member.name.endswith('.mp3'):
                continue
            track_id = 'track_' + os.path.basename(member.name).split('.')[0].zfill(7)
            if track_id not in split_ids:
                continue
            tar.extract(member, AUDIO_TMP)
            audio_path = os.path.join(AUDIO_TMP, member.name)
            try:
                audio = loader_cls(filename=audio_path, sampleRate=16000, resampleQuality=4)()   # mono, 16 kHz
                if EXCERPT_SECONDS is not None and len(audio) > EXCERPT_SECONDS * 16000:
                    mid, half = len(audio) // 2, EXCERPT_SECONDS * 16000 // 2
                    audio = audio[mid - half: mid + half]      # optional: middle excerpt only
                patches = np.asarray(extractor(audio))     # (n_patches, 200)
                ids.append(track_id)
                embs.append(patches.mean(axis=0))          # average over patches -> (200,)
            except Exception as e:                         # report unreadable files instead of stopping
                print('   could not process', track_id, '->', e)
            os.remove(audio_path)
    emb = np.stack(embs) if embs else np.zeros((0, 200), dtype=np.float32)
    np.savez(out_file, ids=np.array(ids), emb=emb)
    os.remove(tar_path)
    return f'{len(ids)} tracks'

if RUN_AUDIO:
    import essentia.standard as es
    if not os.path.exists(MODEL_PB):
        urllib.request.urlretrieve('https://essentia.upf.edu/models/autotagging/msd/msd-musicnn-1.pb', MODEL_PB)
    extractor = es.TensorflowPredictMusiCNN(graphFilename=MODEL_PB, output='model/dense/BiasAdd')
    with open(TAR_LIST) as f:
        tar_entries = [line.split() for line in f if line.strip()]
    if MAX_ARCHIVES is not None:                       # quick smoke test on the first archive(s)
        tar_entries = tar_entries[:MAX_ARCHIVES]
    for k, (expected_sha, tar_name) in enumerate(tar_entries, 1):
        print(f'[{k}/{len(tar_entries)}] {tar_name}:', embed_archive(tar_name, expected_sha, extractor, es.MonoLoader))
else:
    print('RUN_AUDIO = False -> skipping audio download and embedding extraction.')

[1/100] autotagging_moodtheme_audio-low-00.tar: already done
[2/100] autotagging_moodtheme_audio-low-01.tar: already done
[3/100] autotagging_moodtheme_audio-low-02.tar: already done
[4/100] autotagging_moodtheme_audio-low-03.tar: already done
[5/100] autotagging_moodtheme_audio-low-04.tar: already done
[6/100] autotagging_moodtheme_audio-low-05.tar: already done
[7/100] autotagging_moodtheme_audio-low-06.tar: already done
[8/100] autotagging_moodtheme_audio-low-07.tar: already done
[9/100] autotagging_moodtheme_audio-low-08.tar: already done
[10/100] autotagging_moodtheme_audio-low-09.tar: already done
[11/100] autotagging_moodtheme_audio-low-10.tar: already done
[12/100] autotagging_moodtheme_audio-low-11.tar: already done
[13/100] autotagging_moodtheme_audio-low-12.tar: already done
[14/100] autotagging_moodtheme_audio-low-13.tar: already done
[15/100] autotagging_moodtheme_audio-low-14.tar: already done
[16/100] autotagging_moodtheme_audio-low-15.tar: already done
[17/100] autotagg

[26/100] autotagging_moodtheme_audio-low-25.tar: already done
[27/100] autotagging_moodtheme_audio-low-26.tar: already done
[28/100] autotagging_moodtheme_audio-low-27.tar: already done
[29/100] autotagging_moodtheme_audio-low-28.tar: already done
[30/100] autotagging_moodtheme_audio-low-29.tar: already done
[31/100] autotagging_moodtheme_audio-low-30.tar: already done
[32/100] autotagging_moodtheme_audio-low-31.tar: already done
[33/100] autotagging_moodtheme_audio-low-32.tar: already done
[34/100] autotagging_moodtheme_audio-low-33.tar: already done
[35/100] autotagging_moodtheme_audio-low-34.tar: already done
[36/100] autotagging_moodtheme_audio-low-35.tar: already done
[37/100] autotagging_moodtheme_audio-low-36.tar: already done
[38/100] autotagging_moodtheme_audio-low-37.tar: already done
[39/100] autotagging_moodtheme_audio-low-38.tar: already done
[40/100] autotagging_moodtheme_audio-low-39.tar: already done


[41/100] autotagging_moodtheme_audio-low-40.tar: already done
[42/100] autotagging_moodtheme_audio-low-41.tar: already done
[43/100] autotagging_moodtheme_audio-low-42.tar: already done
[44/100] autotagging_moodtheme_audio-low-43.tar: already done
[45/100] autotagging_moodtheme_audio-low-44.tar: already done
[46/100] autotagging_moodtheme_audio-low-45.tar: already done
[47/100] autotagging_moodtheme_audio-low-46.tar: already done
[48/100] autotagging_moodtheme_audio-low-47.tar: already done
[49/100] autotagging_moodtheme_audio-low-48.tar: already done
[50/100] autotagging_moodtheme_audio-low-49.tar: already done
[51/100] autotagging_moodtheme_audio-low-50.tar: already done
[52/100] autotagging_moodtheme_audio-low-51.tar: already done
[53/100] autotagging_moodtheme_audio-low-52.tar: already done
[54/100] autotagging_moodtheme_audio-low-53.tar: already done
[55/100] autotagging_moodtheme_audio-low-54.tar: already done
[56/100] autotagging_moodtheme_audio-low-55.tar: already done
[57/100]

[64/100] autotagging_moodtheme_audio-low-63.tar: already done
[65/100] autotagging_moodtheme_audio-low-64.tar: already done
[66/100] autotagging_moodtheme_audio-low-65.tar: already done
[67/100] autotagging_moodtheme_audio-low-66.tar: already done
[68/100] autotagging_moodtheme_audio-low-67.tar: already done
[69/100] autotagging_moodtheme_audio-low-68.tar: already done
[70/100] autotagging_moodtheme_audio-low-69.tar: already done
[71/100] autotagging_moodtheme_audio-low-70.tar: already done
[72/100] autotagging_moodtheme_audio-low-71.tar: already done
[73/100] autotagging_moodtheme_audio-low-72.tar: already done
[74/100] autotagging_moodtheme_audio-low-73.tar: already done
[75/100] autotagging_moodtheme_audio-low-74.tar: already done
[76/100] autotagging_moodtheme_audio-low-75.tar: already done
[77/100] autotagging_moodtheme_audio-low-76.tar: already done
[78/100] autotagging_moodtheme_audio-low-77.tar: already done
[79/100] autotagging_moodtheme_audio-low-78.tar: already done
[80/100]

[92/100] autotagging_moodtheme_audio-low-91.tar: already done
[93/100] autotagging_moodtheme_audio-low-92.tar: already done
[94/100] autotagging_moodtheme_audio-low-93.tar: already done
[95/100] autotagging_moodtheme_audio-low-94.tar: already done
[96/100] autotagging_moodtheme_audio-low-95.tar: already done
[97/100] autotagging_moodtheme_audio-low-96.tar: already done
[98/100] autotagging_moodtheme_audio-low-97.tar: already done
[99/100] autotagging_moodtheme_audio-low-98.tar: already done
[100/100] autotagging_moodtheme_audio-low-99.tar: already done


In [18]:
# Collect the saved embeddings into one matrix per partition (same row order as the label matrices).
emb_files = sorted(f for f in os.listdir(EMB_DIR) if f.endswith('.npz'))
if emb_files:
    emb_lookup = {}
    for f in emb_files:
        z = np.load(os.path.join(EMB_DIR, f))
        emb_lookup.update(zip(z['ids'], z['emb']))
    missing = {s: int((~d['TRACK_ID'].isin(emb_lookup.keys())).sum()) for s, d in splits.items()}
    print('Embeddings loaded:', len(emb_lookup), '| missing per partition:', missing)
    AUDIO_READY = all(v == 0 for v in missing.values())
    if AUDIO_READY:
        X_audio = {s: np.stack([emb_lookup[t] for t in d['TRACK_ID']]) for s, d in splits.items()}
        print({s: X.shape for s, X in X_audio.items()})
else:
    AUDIO_READY = False
    print('No audio embeddings yet -> audio-only and fusion results are skipped.')

Embeddings loaded: 17982 | missing per partition: {'train': 0, 'validation': 0, 'test': 0}
{'train': (9949, 200), 'validation': (3802, 200), 'test': (4231, 200)}


## 7. Audio-only baseline and late fusion
The audio classifier is the same one-vs-rest logistic regression as the text branch (L2, C = 1.0, liblinear, max 1,000 iterations, random state 42), with the 200-d embeddings
standardised (zero mean, unit variance, fitted on training data only). Late fusion averages the two branches' probabilities:
$P_{\text{fusion}}(i,j) = \tfrac{1}{2}\,[P_{\text{audio}}(i,j) + P_{\text{text}}(i,j)]$.

In [19]:
if AUDIO_READY:
    audio_model = OneVsRestClassifier(make_pipeline(
        StandardScaler(),
        LogisticRegression(C=LR_C, penalty='l2', solver='liblinear', max_iter=1000, random_state=SEED)), n_jobs=-1)
    audio_model.fit(X_audio['train'], Y['train'])
    P_audio_val = audio_model.predict_proba(X_audio['validation'])
    P_fusion_val = (P_audio_val + P_text_val) / 2                       # Equation (1)

    val_table = pd.DataFrame({
        'Audio-only': evaluate(Y['validation'], P_audio_val),
        'Text-only': text_val_results,
        'Late fusion': evaluate(Y['validation'], P_fusion_val),
    }).T
    print('VALIDATION results')
    display(val_table.round(4))
    per_tag_audio = {short(official_tags[j]): average_precision_score(Y['validation'][:, j], P_audio_val[:, j])
                     for j in range(len(official_tags))}
    per_tag['Audio AP'] = per_tag['Tag'].map(per_tag_audio)
    per_tag['Fusion AP'] = per_tag['Tag'].map({short(official_tags[j]): average_precision_score(Y['validation'][:, j], P_fusion_val[:, j])
                                               for j in range(len(official_tags))})
    per_tag['Fusion - Audio'] = per_tag['Fusion AP'] - per_tag['Audio AP']
    print('Tags where fusion helps most / hurts most (validation):')
    display(per_tag.sort_values('Fusion - Audio', ascending=False).head(8).round(4))
    display(per_tag.sort_values('Fusion - Audio').head(5).round(4))
else:
    print('Audio embeddings not available yet -> skipped.')

VALIDATION results


,PR-AUC-macro,PR-AUC-micro,ROC-AUC-macro,PR-AUC-macro (uncommon 14),PR-AUC-macro (other 42)
Audio-only,0.0961,0.1026,0.7190,0.0877,0.0990
Text-only,0.1163,0.1210,0.6281,0.0991,0.1220
Late fusion,0.1317,0.1392,0.7434,0.1259,0.1336


Tags where fusion helps most / hurts most (validation):


,Tag,Training count,Validation count,Uncommon,Text AP,Audio AP,Fusion AP,Fusion - Audio
36,christmas,368,74,False,0.7441,0.1111,0.6263,0.5152
13,trailer,159,71,True,0.5887,0.1988,0.5080,0.3092
4,holiday,98,35,True,0.3650,0.0193,0.2111,0.1918
32,corporate,349,118,False,0.3375,0.1872,0.3648,0.1776
37,motivational,372,75,False,0.2339,0.1420,0.2706,0.1286
48,inspiring,561,136,False,0.1368,0.0658,0.1331,0.0672
33,space,362,93,False,0.2903,0.0681,0.1284,0.0604
26,summer,261,91,False,0.1587,0.2265,0.2861,0.0595


,Tag,Training count,Validation count,Uncommon,Text AP,Audio AP,Fusion AP,Fusion - Audio
23,fun,220,187,False,0.0479,0.1313,0.1170,-0.0143
53,relaxing,763,278,False,0.0925,0.1997,0.1922,-0.0076
43,deep,429,69,False,0.0232,0.1574,0.1503,-0.0071
18,dramatic,195,34,False,0.0132,0.0319,0.0259,-0.0060
54,energetic,857,233,False,0.0982,0.1579,0.1540,-0.0039


## 8. Success criteria and final test evaluation
Success criteria (paper Section 4.2), checked on validation during development and once on test at the end:
1. Our audio-only model reaches at least the official VGG-ish baseline (PR-AUC-macro 0.108, test).
2. Late fusion improves uncommon-tag PR-AUC-macro by at least **0.02** over our audio-only model.
3. The 95% paired-bootstrap confidence interval (1,000 resamples of tracks) for that improvement excludes zero.
4. Late fusion does not lower overall PR-AUC-macro relative to the audio-only model.

In [20]:
def check_success(y_true, p_audio, p_fusion, label):
    audio_res, fusion_res = evaluate(y_true, p_audio), evaluate(y_true, p_fusion)
    diff, low, high = paired_bootstrap(y_true, p_fusion, p_audio, uncommon_idx)
    checks = {
        '1. Audio-only PR-AUC-macro >= 0.108': audio_res['PR-AUC-macro'] >= 0.108,
        '2. Uncommon-tag gain >= 0.02': diff >= 0.02,
        '3. 95% CI of gain excludes 0': low > 0,
        '4. Overall PR-AUC-macro not lower': fusion_res['PR-AUC-macro'] >= audio_res['PR-AUC-macro'],
    }
    print(f'{label}: uncommon-tag gain = {diff:.4f} (95% CI {low:.4f} to {high:.4f})')
    for name, passed in checks.items():
        print(f'  {"PASS" if passed else "FAIL"}  {name}')
    return checks

if AUDIO_READY:
    check_success(Y['validation'], P_audio_val, P_fusion_val, 'VALIDATION')
else:
    print('Audio embeddings not available yet -> success criteria not evaluated.')

VALIDATION: uncommon-tag gain = 0.0382 (95% CI 0.0282 to 0.0504)
  FAIL  1. Audio-only PR-AUC-macro >= 0.108
  PASS  2. Uncommon-tag gain >= 0.02
  PASS  3. 95% CI of gain excludes 0
  PASS  4. Overall PR-AUC-macro not lower


In [21]:
# ONE-TIME test evaluation. Only switch RUN_FINAL_TEST on after every model choice is frozen.
if RUN_FINAL_TEST and AUDIO_READY:
    P_text_test = text_model.predict_proba(X_text['test'])
    P_audio_test = audio_model.predict_proba(X_audio['test'])
    P_fusion_test = (P_audio_test + P_text_test) / 2
    test_table = pd.DataFrame({
        'Audio-only': evaluate(Y['test'], P_audio_test),
        'Text-only': evaluate(Y['test'], P_text_test),
        'Late fusion': evaluate(Y['test'], P_fusion_test),
    }).T
    print('TEST results')
    display(test_table.round(4))
    check_success(Y['test'], P_audio_test, P_fusion_test, 'TEST')
else:
    print('Test split not used (RUN_FINAL_TEST is False or audio not ready).')

Test split not used (RUN_FINAL_TEST is False or audio not ready).


In [22]:
# Save models, predictions, and tables so the numbers in the paper can be checked later.
joblib.dump(text_model, os.path.join(OUTPUT_DIR, 'text_model.joblib'))
joblib.dump(tfidf, os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.joblib'))
joblib.dump(official_tags, os.path.join(OUTPUT_DIR, 'tag_list.joblib'))
np.save(os.path.join(OUTPUT_DIR, 'text_validation_probs.npy'), P_text_val)
np.save(os.path.join(OUTPUT_DIR, 'validation_labels.npy'), Y['validation'])
per_tag.to_csv(os.path.join(OUTPUT_DIR, 'per_tag_validation_results.csv'), index=False)
with open(os.path.join(OUTPUT_DIR, 'run_info.json'), 'w') as f:
    json.dump({'versions': versions, 'dataset_commit': DATASET_COMMIT, 'seed': SEED,
               'uncommon_tags': [short(official_tags[i]) for i in uncommon_idx],
               'text_validation': text_val_results}, f, indent=2)
if AUDIO_READY:
    joblib.dump(audio_model, os.path.join(OUTPUT_DIR, 'audio_model.joblib'))
    np.save(os.path.join(OUTPUT_DIR, 'audio_validation_probs.npy'), P_audio_val)
    val_table.to_csv(os.path.join(OUTPUT_DIR, 'validation_results.csv'))
# Results table in LaTeX, ready to paste into the paper (Table 2)
def latex_table(table, caption):
    rows = [f"{name} & " + ' & '.join(f'{table.loc[name, c]:.4f}' for c in
            ['PR-AUC-macro', 'PR-AUC-micro', 'ROC-AUC-macro', 'PR-AUC-macro (uncommon 14)', 'PR-AUC-macro (other 42)']) + r' \\'
            for name in table.index]
    return '\n'.join(rows) + f'\n% {caption}'
if AUDIO_READY:
    with open(os.path.join(OUTPUT_DIR, 'results_table_validation.tex'), 'w') as f:
        f.write(latex_table(val_table, 'validation'))
    if RUN_FINAL_TEST:
        with open(os.path.join(OUTPUT_DIR, 'results_table_test.tex'), 'w') as f:
            f.write(latex_table(test_table, 'test'))
print('Saved to', OUTPUT_DIR, ':', sorted(os.listdir(OUTPUT_DIR)))

Saved to /mnt/d/FLAIR-Group-4/flair-mood-tagging/FLAIR_Group_4_results : ['audio_model.joblib', 'audio_validation_probs.npy', 'musicnn_embeddings', 'per_tag_validation_results.csv', 'results_table_validation.tex', 'run_info.json', 'tag_list.joblib', 'text_model.joblib', 'text_validation_probs.npy', 'tfidf_vectorizer.joblib', 'validation_labels.npy', 'validation_results.csv']
